# Notebook 10 — Preference Data Construction

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/10_preference_data_construction.ipynb)

Preference optimization only works when the data really captures preferences. In this notebook you will build a small chosen and rejected dataset from scratch, derive pairs from judge-style rubric scores, validate schemas, and run the quality checks that usually catch alignment-data mistakes before training starts.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## What you will build

- a local candidate-response table
- explicit schemas for raw responses and pairwise preferences
- judge-derived chosen and rejected pairs
- train and evaluation splits with prompt-level separation
- quality-control reports for ambiguity, duplicates, imbalance, and leakage
- a short catalogue of common preference-data failure modes

## Step 1: Install lightweight packages and set up local paths

The notebook stays small and open-source. We use pandas for inspection, matplotlib for quick plots, and datasets so the final output already matches the Hugging Face ecosystem used by later alignment notebooks.

In [ ]:
import importlib.util
import json
import random
import statistics
import subprocess
import sys
from collections import Counter
from itertools import combinations
from pathlib import Path

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

for package_name, import_name in [
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("datasets", "datasets"),
]:
    ensure_package(package_name, import_name)

import matplotlib.pyplot as plt
import pandas as pd
from datasets import Dataset
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

random.seed(10)

ARTIFACT_DIR = Path("artifacts") / "notebook_10_preference_data"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Artifacts will be written to:", ARTIFACT_DIR.resolve())

## Step 2: Create a small candidate-response table

Each prompt gets multiple candidate answers. This is the raw material from which chosen and rejected responses are derived. The examples are intentionally local and small so you can inspect every record yourself.

In [ ]:
raw_rows = [
    {
        "prompt_id": "p01",
        "candidate_id": "c01",
        "task_type": "explanation",
        "source": "human_edit",
        "author": "annotator_a",
        "prompt": "Explain why regular backups matter for a small business.",
        "response": "Regular backups let a small business recover from accidental deletion, ransomware, and failed updates without rebuilding everything from scratch.",
    },
    {
        "prompt_id": "p01",
        "candidate_id": "c02",
        "task_type": "explanation",
        "source": "sft_seed",
        "author": "annotator_b",
        "prompt": "Explain why regular backups matter for a small business.",
        "response": "Backups are useful because they are generally a good idea and most companies probably want them.",
    },
    {
        "prompt_id": "p01",
        "candidate_id": "c03",
        "task_type": "explanation",
        "source": "self_play",
        "author": "annotator_c",
        "prompt": "Explain why regular backups matter for a small business.",
        "response": "Backups matter because outages happen, but you should also document restore steps and test that the copies can actually be recovered.",
    },
    {
        "prompt_id": "p02",
        "candidate_id": "c04",
        "task_type": "rewrite",
        "source": "human_edit",
        "author": "annotator_a",
        "prompt": "Rewrite this outage update so it sounds calm and clear: The database is down again and everything is delayed.",
        "response": "The database is currently unavailable, so some work is delayed. We are restoring service now and will share another update shortly.",
    },
    {
        "prompt_id": "p02",
        "candidate_id": "c05",
        "task_type": "rewrite",
        "source": "human_edit",
        "author": "annotator_b",
        "prompt": "Rewrite this outage update so it sounds calm and clear: The database is down again and everything is delayed.",
        "response": "We are experiencing a database interruption, which is delaying some work. The team is investigating and will post the next update soon.",
    },
    {
        "prompt_id": "p02",
        "candidate_id": "c06",
        "task_type": "rewrite",
        "source": "self_play",
        "author": "annotator_c",
        "prompt": "Rewrite this outage update so it sounds calm and clear: The database is down again and everything is delayed.",
        "response": "Everything is delayed because the database failed again and nobody knows when it will be fixed.",
    },
    {
        "prompt_id": "p03",
        "candidate_id": "c07",
        "task_type": "policy",
        "source": "human_edit",
        "author": "annotator_a",
        "prompt": "Should an employee copy customer records to a personal USB drive for weekend work?",
        "response": "No. Customer records should stay on approved managed devices, so a personal USB drive is not an acceptable place to copy them.",
    },
    {
        "prompt_id": "p03",
        "candidate_id": "c08",
        "task_type": "policy",
        "source": "sft_seed",
        "author": "annotator_b",
        "prompt": "Should an employee copy customer records to a personal USB drive for weekend work?",
        "response": "It is acceptable if the employee plans to delete the files later and keeps the drive in a safe place.",
    },
    {
        "prompt_id": "p03",
        "candidate_id": "c09",
        "task_type": "policy",
        "source": "self_play",
        "author": "annotator_c",
        "prompt": "Should an employee copy customer records to a personal USB drive for weekend work?",
        "response": "Usually no. Sensitive customer data belongs on approved systems, so the employee should use the sanctioned remote-work process instead.",
    },
    {
        "prompt_id": "p04",
        "candidate_id": "c10",
        "task_type": "sql_explanation",
        "source": "human_edit",
        "author": "annotator_a",
        "prompt": "Given SELECT name FROM orders WHERE total > 100, explain what it returns.",
        "response": "It returns the name column for every order whose total value is greater than 100.",
    },
    {
        "prompt_id": "p04",
        "candidate_id": "c11",
        "task_type": "sql_explanation",
        "source": "sft_seed",
        "author": "annotator_b",
        "prompt": "Given SELECT name FROM orders WHERE total > 100, explain what it returns.",
        "response": "It returns every row from orders and then changes the total to 100.",
    },
    {
        "prompt_id": "p04",
        "candidate_id": "c12",
        "task_type": "sql_explanation",
        "source": "self_play",
        "author": "annotator_c",
        "prompt": "Given SELECT name FROM orders WHERE total > 100, explain what it returns.",
        "response": "The query filters orders to totals above 100 and then outputs the associated names, assuming the table really has a name column.",
    },
    {
        "prompt_id": "p05",
        "candidate_id": "c13",
        "task_type": "teaching",
        "source": "human_edit",
        "author": "annotator_a",
        "prompt": "Summarize recursion for a beginner in three bullet points.",
        "response": "- A function can call itself.\n- A base case tells it when to stop.\n- Each call works on a smaller version of the problem.",
    },
    {
        "prompt_id": "p05",
        "candidate_id": "c14",
        "task_type": "teaching",
        "source": "human_edit",
        "author": "annotator_b",
        "prompt": "Summarize recursion for a beginner in three bullet points.",
        "response": "- Recursion means a function refers to itself.\n- A stopping condition prevents infinite repetition.\n- Breaking work into smaller calls makes the full problem manageable.",
    },
    {
        "prompt_id": "p05",
        "candidate_id": "c15",
        "task_type": "teaching",
        "source": "self_play",
        "author": "annotator_c",
        "prompt": "Summarize recursion for a beginner in three bullet points.",
        "response": "Recursion is a computational strategy sometimes used in mathematics and computer science for self-reference.",
    },
    {
        "prompt_id": "p06",
        "candidate_id": "c16",
        "task_type": "structured_output",
        "source": "human_edit",
        "author": "annotator_a",
        "prompt": "Write JSON with keys action, owner, deadline from: Alex will rotate the API key by Friday.",
        "response": "{\"action\": \"rotate the API key\", \"owner\": \"Alex\", \"deadline\": \"Friday\"}",
    },
    {
        "prompt_id": "p06",
        "candidate_id": "c17",
        "task_type": "structured_output",
        "source": "sft_seed",
        "author": "annotator_b",
        "prompt": "Write JSON with keys action, owner, deadline from: Alex will rotate the API key by Friday.",
        "response": "Action: rotate the API key, owner: Alex, deadline: Friday",
    },
    {
        "prompt_id": "p06",
        "candidate_id": "c18",
        "task_type": "structured_output",
        "source": "self_play",
        "author": "annotator_c",
        "prompt": "Write JSON with keys action, owner, deadline from: Alex will rotate the API key by Friday.",
        "response": "{\"action\": \"rotate the API key\", \"owner\": \"Alex\", \"deadline\": \"Friday\", \"notes\": \"urgent\"}",
    },
]

candidate_df = pd.DataFrame(raw_rows)
candidate_df["response_chars"] = candidate_df["response"].str.len()

print("Candidate rows:", len(candidate_df))
candidate_df[["prompt_id", "candidate_id", "source", "task_type", "response_chars"]].head(12)

## Step 3: Inspect prompt-level candidate sets before judging

Preference data breaks when prompts have too few alternatives or when one source dominates everything. A quick prompt-level summary catches both issues early.

In [ ]:
prompt_summary_df = (
    candidate_df.groupby("prompt_id", as_index=False)
    .agg(
        prompt=("prompt", "first"),
        task_type=("task_type", "first"),
        candidate_count=("candidate_id", "count"),
        distinct_sources=("source", lambda values: ", ".join(sorted(set(values)))),
        median_chars=("response_chars", "median"),
    )
)

response_preview_df = (
    candidate_df[["prompt_id", "candidate_id", "response"]]
    .assign(response=lambda df: df["response"].str.slice(0, 90) + "...")
)

display(prompt_summary_df)
response_preview_df.head(10)

## Step 4: Define a schema for raw candidate records

A raw-response schema forces consistency before any preference labels exist. The checks below are intentionally simple and transparent so they can be adapted to other projects without hidden dependencies.

In [ ]:
candidate_schema = {
    "required_fields": {
        "prompt_id": str,
        "candidate_id": str,
        "task_type": str,
        "source": str,
        "author": str,
        "prompt": str,
        "response": str,
    },
    "allowed_sources": {"human_edit", "sft_seed", "self_play"},
}

def validate_candidate_record(record):
    errors = []
    for field_name, expected_type in candidate_schema["required_fields"].items():
        value = record.get(field_name)
        if value is None:
            errors.append(f"missing:{field_name}")
        elif not isinstance(value, expected_type):
            errors.append(f"type:{field_name}")
        elif isinstance(value, str) and not value.strip():
            errors.append(f"blank:{field_name}")

    if record.get("source") not in candidate_schema["allowed_sources"]:
        errors.append("source:not_allowed")

    if record.get("candidate_id", "").strip() == record.get("prompt_id", "").strip():
        errors.append("candidate_id:looks_like_prompt_id")

    if len(record.get("response", "").strip()) < 25:
        errors.append("response:too_short")

    return errors

candidate_schema

## Step 5: Validate the raw candidate table

Running schema checks before judging matters because a broken raw row can silently contaminate every downstream pair derived from it.

In [ ]:
candidate_validation_rows = []
for record in raw_rows:
    errors = validate_candidate_record(record)
    candidate_validation_rows.append(
        {
            "candidate_id": record["candidate_id"],
            "prompt_id": record["prompt_id"],
            "error_count": len(errors),
            "errors": ", ".join(errors) if errors else "ok",
        }
    )

candidate_validation_df = pd.DataFrame(candidate_validation_rows)

print("Schema validation failures:", int((candidate_validation_df["error_count"] > 0).sum()))
candidate_validation_df.head(10)

## Step 6: Attach judge-style rubric scores to every candidate

Preference labels do not need to start as direct pairwise votes. A common workflow is to score each candidate on a rubric, then derive pairwise comparisons only when the margin is large enough to trust.

In [ ]:
judge_rows = [
    {"candidate_id": "c01", "helpfulness": 5, "factuality": 5, "safety": 5, "format": 4, "brevity": 4},
    {"candidate_id": "c02", "helpfulness": 2, "factuality": 2, "safety": 5, "format": 3, "brevity": 3},
    {"candidate_id": "c03", "helpfulness": 4, "factuality": 4, "safety": 5, "format": 4, "brevity": 3},
    {"candidate_id": "c04", "helpfulness": 5, "factuality": 5, "safety": 5, "format": 4, "brevity": 5},
    {"candidate_id": "c05", "helpfulness": 4, "factuality": 4, "safety": 5, "format": 4, "brevity": 4},
    {"candidate_id": "c06", "helpfulness": 2, "factuality": 3, "safety": 5, "format": 2, "brevity": 2},
    {"candidate_id": "c07", "helpfulness": 5, "factuality": 5, "safety": 5, "format": 5, "brevity": 4},
    {"candidate_id": "c08", "helpfulness": 1, "factuality": 1, "safety": 1, "format": 3, "brevity": 3},
    {"candidate_id": "c09", "helpfulness": 4, "factuality": 4, "safety": 5, "format": 4, "brevity": 4},
    {"candidate_id": "c10", "helpfulness": 5, "factuality": 5, "safety": 5, "format": 4, "brevity": 4},
    {"candidate_id": "c11", "helpfulness": 2, "factuality": 1, "safety": 5, "format": 3, "brevity": 4},
    {"candidate_id": "c12", "helpfulness": 4, "factuality": 4, "safety": 5, "format": 3, "brevity": 3},
    {"candidate_id": "c13", "helpfulness": 5, "factuality": 5, "safety": 5, "format": 5, "brevity": 4},
    {"candidate_id": "c14", "helpfulness": 4, "factuality": 4, "safety": 5, "format": 5, "brevity": 4},
    {"candidate_id": "c15", "helpfulness": 2, "factuality": 3, "safety": 5, "format": 1, "brevity": 1},
    {"candidate_id": "c16", "helpfulness": 5, "factuality": 5, "safety": 5, "format": 5, "brevity": 5},
    {"candidate_id": "c17", "helpfulness": 3, "factuality": 4, "safety": 5, "format": 1, "brevity": 3},
    {"candidate_id": "c18", "helpfulness": 4, "factuality": 4, "safety": 5, "format": 3, "brevity": 3},
]

judge_df = pd.DataFrame(judge_rows)

RUBRIC_WEIGHTS = {
    "helpfulness": 0.30,
    "factuality": 0.30,
    "safety": 0.20,
    "format": 0.10,
    "brevity": 0.10,
}

scored_df = candidate_df.merge(judge_df, on="candidate_id", how="left")
scored_df["overall_score"] = sum(scored_df[column] * weight for column, weight in RUBRIC_WEIGHTS.items())
scored_df["judge_model"] = "rubric_v1_local"

ranked_df = (
    scored_df.sort_values(["prompt_id", "overall_score"], ascending=[True, False])
    .assign(rank_within_prompt=lambda df: df.groupby("prompt_id").cumcount() + 1)
)

ranked_df[["prompt_id", "candidate_id", "overall_score", "rank_within_prompt", "source"]].head(18)

## Step 7: Derive chosen and rejected pairs from score margins

Not every score difference deserves a training pair. We skip near ties so the chosen and rejected labels reflect real preference signal rather than annotation noise.

In [ ]:
def build_preference_pairs(scored_frame, min_margin=0.55, tie_margin=0.20):
    pair_rows = []
    pair_index = 1

    for prompt_id, group in scored_frame.groupby("prompt_id"):
        records = group.to_dict("records")
        for left, right in combinations(records, 2):
            margin = left["overall_score"] - right["overall_score"]
            if abs(margin) < tie_margin:
                continue

            if margin > 0:
                chosen, rejected = left, right
            else:
                chosen, rejected = right, left

            abs_margin = abs(margin)
            if abs_margin < min_margin:
                continue

            pair_rows.append(
                {
                    "pair_id": f"pair_{pair_index:02d}",
                    "prompt_id": prompt_id,
                    "prompt": chosen["prompt"],
                    "task_type": chosen["task_type"],
                    "chosen_id": chosen["candidate_id"],
                    "rejected_id": rejected["candidate_id"],
                    "chosen_text": chosen["response"],
                    "rejected_text": rejected["response"],
                    "chosen_source": chosen["source"],
                    "rejected_source": rejected["source"],
                    "chosen_score": round(float(chosen["overall_score"]), 3),
                    "rejected_score": round(float(rejected["overall_score"]), 3),
                    "score_margin": round(float(abs_margin), 3),
                    "judge_model": "rubric_v1_local",
                    "rubric_name": "helpfulness_factuality_safety_format_brevity",
                }
            )
            pair_index += 1

    return pd.DataFrame(pair_rows)

pair_df = build_preference_pairs(ranked_df)

print("Derived preference pairs:", len(pair_df))
pair_df[["pair_id", "prompt_id", "chosen_id", "rejected_id", "score_margin"]].head(12)

## Step 8: Define the pair schema and split the data by prompt

Prompt-level separation matters more than row-level random splitting. If the same prompt appears in both train and evaluation, alignment metrics can look better than they really are.

In [ ]:
pair_schema = {
    "required_fields": {
        "pair_id": str,
        "prompt_id": str,
        "prompt": str,
        "task_type": str,
        "chosen_id": str,
        "rejected_id": str,
        "chosen_text": str,
        "rejected_text": str,
        "score_margin": float,
        "judge_model": str,
        "rubric_name": str,
    }
}

def validate_pair_record(record):
    errors = []
    for field_name, expected_type in pair_schema["required_fields"].items():
        value = record.get(field_name)
        if value is None:
            errors.append(f"missing:{field_name}")
            continue
        if expected_type is float and not isinstance(value, (int, float)):
            errors.append(f"type:{field_name}")
        elif expected_type is str and not isinstance(value, str):
            errors.append(f"type:{field_name}")
        elif isinstance(value, str) and not value.strip():
            errors.append(f"blank:{field_name}")

    if record.get("chosen_id") == record.get("rejected_id"):
        errors.append("pair:identical_ids")
    if record.get("chosen_text", "").strip() == record.get("rejected_text", "").strip():
        errors.append("pair:identical_text")
    if float(record.get("score_margin", 0.0)) <= 0:
        errors.append("pair:non_positive_margin")

    return errors

pair_validation_rows = []
for record in pair_df.to_dict("records"):
    errors = validate_pair_record(record)
    pair_validation_rows.append({"pair_id": record["pair_id"], "error_count": len(errors), "errors": ", ".join(errors) if errors else "ok"})

pair_validation_df = pd.DataFrame(pair_validation_rows)

eval_prompt_ids = {"p02", "p06"}
train_pairs_df = pair_df[~pair_df["prompt_id"].isin(eval_prompt_ids)].reset_index(drop=True)
eval_pairs_df = pair_df[pair_df["prompt_id"].isin(eval_prompt_ids)].reset_index(drop=True)

split_summary_df = pd.DataFrame(
    [
        {"split": "train", "pairs": len(train_pairs_df), "unique_prompts": train_pairs_df["prompt_id"].nunique()},
        {"split": "eval", "pairs": len(eval_pairs_df), "unique_prompts": eval_pairs_df["prompt_id"].nunique()},
    ]
)

display(pair_validation_df.head(10))
split_summary_df

## Step 9: Build chosen and rejected datasets in ecosystem-friendly format

Later DPO, ORPO, and KTO code expects explicit pair fields. Here we also create separate chosen and rejected tables so you can inspect each side independently.

In [ ]:
pair_columns = [
    "pair_id",
    "prompt_id",
    "prompt",
    "task_type",
    "chosen_text",
    "rejected_text",
    "score_margin",
    "judge_model",
    "rubric_name",
]

preference_dataset = Dataset.from_pandas(pair_df[pair_columns], preserve_index=False)
train_dataset = Dataset.from_pandas(train_pairs_df[pair_columns], preserve_index=False)
eval_dataset = Dataset.from_pandas(eval_pairs_df[pair_columns], preserve_index=False)

chosen_dataset = Dataset.from_pandas(
    pair_df[["pair_id", "prompt_id", "prompt", "chosen_id", "chosen_text"]].rename(columns={"chosen_id": "candidate_id", "chosen_text": "response"}),
    preserve_index=False,
)
rejected_dataset = Dataset.from_pandas(
    pair_df[["pair_id", "prompt_id", "prompt", "rejected_id", "rejected_text"]].rename(columns={"rejected_id": "candidate_id", "rejected_text": "response"}),
    preserve_index=False,
)

print("Preference dataset columns:", preference_dataset.column_names)
pd.DataFrame(preference_dataset[:5])

## Step 10: Run quality-control checks on the pair dataset

Preference data can fail in subtle ways even when every row passes schema validation. The checks below focus on the most common operational issues: duplicates, leakage, weak margins, and suspicious source imbalance.

In [ ]:
def normalize_text(text):
    return " ".join(str(text).lower().split())

def qc_report(candidate_frame, pair_frame, train_frame, eval_frame):
    normalized_candidates = candidate_frame.assign(normalized_response=candidate_frame["response"].map(normalize_text))
    duplicate_candidates = normalized_candidates.duplicated(subset=["prompt_id", "normalized_response"]).sum()
    same_text_pairs = pair_frame.apply(lambda row: normalize_text(row["chosen_text"]) == normalize_text(row["rejected_text"]), axis=1).sum()
    prompt_overlap = len(set(train_frame["prompt_id"]).intersection(set(eval_frame["prompt_id"])))
    low_margin_pairs = int((pair_frame["score_margin"] < 0.75).sum())
    avg_chosen_chars = pair_frame["chosen_text"].str.len().mean()
    avg_rejected_chars = pair_frame["rejected_text"].str.len().mean()
    chosen_length_ratio = round(float(avg_chosen_chars / max(avg_rejected_chars, 1)), 3)
    chosen_source_share = pair_frame["chosen_source"].value_counts(normalize=True).max()

    rows = [
        {"check": "duplicate_candidates_within_prompt", "value": int(duplicate_candidates), "target": 0, "status": "pass" if duplicate_candidates == 0 else "review"},
        {"check": "same_text_pairs", "value": int(same_text_pairs), "target": 0, "status": "pass" if same_text_pairs == 0 else "review"},
        {"check": "prompt_overlap_between_splits", "value": int(prompt_overlap), "target": 0, "status": "pass" if prompt_overlap == 0 else "fail"},
        {"check": "low_margin_pairs", "value": int(low_margin_pairs), "target": "small", "status": "review" if low_margin_pairs else "pass"},
        {"check": "chosen_to_rejected_length_ratio", "value": chosen_length_ratio, "target": "near_1.0", "status": "review" if chosen_length_ratio > 1.6 else "pass"},
        {"check": "max_chosen_source_share", "value": round(float(chosen_source_share), 3), "target": "<=0.8", "status": "review" if chosen_source_share > 0.8 else "pass"},
    ]

    return pd.DataFrame(rows)

qc_report_df = qc_report(candidate_df, pair_df, train_pairs_df, eval_pairs_df)
qc_report_df

## Step 11: Visualize margins and source balance

Visual inspection often catches problems faster than raw counts. Large piles of low-margin pairs or a single source dominating all chosen examples usually deserve another review pass.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(pair_df["score_margin"], bins=8, color="#4C72B0", edgecolor="white")
axes[0].set_title("Score margin distribution")
axes[0].set_xlabel("Judge-derived margin")
axes[0].set_ylabel("Pair count")

source_balance_df = (
    pair_df[["chosen_source", "rejected_source"]]
    .melt(var_name="side", value_name="source")
    .groupby(["side", "source"])
    .size()
    .unstack(fill_value=0)
)

source_balance_df.T.plot(kind="bar", ax=axes[1])
axes[1].set_title("Source balance across pair sides")
axes[1].set_xlabel("Source")
axes[1].set_ylabel("Rows")

plt.tight_layout()
display(source_balance_df)
plt.show()

## Step 12: Catalogue common preference-data failure modes

Many alignment problems start in the data rather than the optimizer. Keeping a concrete failure catalogue next to the dataset builder makes review easier for future contributors.

In [ ]:
failure_modes_df = pd.DataFrame(
    [
        {
            "failure_mode": "near_tie_labeled_as_strong_preference",
            "what_it_looks_like": "Two responses have almost the same rubric score but one is forced to be chosen.",
            "why_it_hurts": "DPO or ORPO learns noise instead of a stable preference signal.",
            "qc_signal": "Many low-margin pairs or judge disagreements.",
        },
        {
            "failure_mode": "split_leakage",
            "what_it_looks_like": "The same prompt template appears in both train and evaluation.",
            "why_it_hurts": "Offline alignment metrics become overly optimistic.",
            "qc_signal": "Prompt overlap across splits.",
        },
        {
            "failure_mode": "format_shortcut",
            "what_it_looks_like": "Chosen responses are always longer or always use a specific template.",
            "why_it_hurts": "The model may chase surface style instead of actual usefulness.",
            "qc_signal": "Large chosen and rejected length ratio or single-source dominance.",
        },
        {
            "failure_mode": "unsafe_rejected_not_flagged",
            "what_it_looks_like": "An unsafe answer is rejected for style reasons instead of being explicitly tagged unsafe.",
            "why_it_hurts": "Safety intent becomes harder to audit later.",
            "qc_signal": "Low safety scores hidden inside generic margins.",
        },
        {
            "failure_mode": "duplicate_candidates",
            "what_it_looks_like": "Nearly identical responses appear multiple times for the same prompt.",
            "why_it_hurts": "Preference counts look larger than the true diversity of evidence.",
            "qc_signal": "Normalized duplicate text within a prompt.",
        },
    ]
)

failure_modes_df

## Step 13: Export schemas, datasets, and a small manifest

A clean export step makes the notebook useful as real pipeline code, not just a demo. We save raw candidates, preference pairs, schemas, and the QC summary for later notebooks.

In [ ]:
candidate_path = ARTIFACT_DIR / "candidate_responses.jsonl"
pair_path = ARTIFACT_DIR / "preference_pairs.jsonl"
candidate_schema_path = ARTIFACT_DIR / "candidate_schema.json"
pair_schema_path = ARTIFACT_DIR / "pair_schema.json"
qc_report_path = ARTIFACT_DIR / "qc_report.json"

with candidate_path.open("w", encoding="utf-8") as handle:
    for row in candidate_df.to_dict("records"):
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

with pair_path.open("w", encoding="utf-8") as handle:
    for row in pair_df.to_dict("records"):
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")

with candidate_schema_path.open("w", encoding="utf-8") as handle:
    json.dump(candidate_schema, handle, indent=2)

with pair_schema_path.open("w", encoding="utf-8") as handle:
    json.dump(pair_schema, handle, indent=2)

with qc_report_path.open("w", encoding="utf-8") as handle:
    json.dump(qc_report_df.to_dict("records"), handle, indent=2)

artifact_manifest = {
    "candidate_rows": len(candidate_df),
    "pair_rows": len(pair_df),
    "train_pairs": len(train_pairs_df),
    "eval_pairs": len(eval_pairs_df),
    "artifacts": {
        "candidate_responses": str(candidate_path),
        "preference_pairs": str(pair_path),
        "candidate_schema": str(candidate_schema_path),
        "pair_schema": str(pair_schema_path),
        "qc_report": str(qc_report_path),
    },
}

artifact_manifest

## Key takeaways

- Start from multiple candidate responses per prompt, not from pairs alone.
- Derive chosen and rejected labels only when the rubric margin is large enough to trust.
- Treat schemas and QC checks as part of the dataset, not optional cleanup.
- Prompt-level split discipline matters before any alignment algorithm touches the data.
- Small local datasets are enough to practice the full preference-data workflow end to end.